# AML transaction monitoring: XGBoost vs graph neural network on IBM HI-Medium

Runs the project's v2 pipeline (rules, XGBoost, durability checks) and then the GNN experiment
(directional GraphSAGE) on the same time windows and the same test accounts.

**Before running**
1. *Add data*: the Kaggle dataset **IBM Transactions for Anti Money Laundering (AML)**.
2. *Add your code*: either push the repo to GitHub and set `REPO_URL` below (needs Internet on),
   or upload the project folder as a Kaggle dataset (without `data/`) and leave `REPO_URL` empty.
3. *Settings → Accelerator*: **GPU T4 x2** or **GPU P100**.
4. Recommended: first run with `DATASET = "HI-Small"` (about 30-40 min) to check everything,
   then switch to `"HI-Medium"` (about 2-3 hours). Use *Save Version → Save & Run All* so it runs
   in the background.

Every number this notebook prints comes from the run itself. `summary_gnn.md` only claims a
GNN gain if the paired bootstrap says it is significant.

In [ ]:
DATASET = "HI-Medium"      # "HI-Small" for a quick check first
REPO_URL = "https://github.com/rishysp/AML"   # your repo (Internet must be on)
N_WINDOWS = {"HI-Small": 5, "HI-Medium": 30}[DATASET]  # ~1M transactions per window in both cases
N_TRIALS = 10              # XGBoost hyperparameter search trials
MAX_ROUNDS = {"HI-Small": 2000, "HI-Medium": 4000}[DATASET]  # HI-Medium hit the 2000 cap last time
MAX_TRAIN_WINDOWS = None   # None = all 28 training windows. If the run is killed again for memory,
                           # set e.g. 15 to train on the 15 most recent windows only

In [ ]:
import glob, os, shutil, subprocess, sys
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("system RAM:", [l.split()[1] for l in open("/proc/meminfo") if l.startswith("MemTotal")][0], "kB")
csv = glob.glob(f"/kaggle/input/**/{DATASET}_Trans.csv", recursive=True)
assert csv, f"{DATASET}_Trans.csv not found: add the IBM AML dataset to this notebook"
RAW = csv[0]
print("data:", RAW, f"{os.path.getsize(RAW) / 1e9:.2f} GB")

REPO = "/kaggle/working/repo"
if REPO_URL and os.path.exists(f"{REPO}/.git"):
    # Repo already here from an earlier run in this session: pull the latest code
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
elif not os.path.exists(REPO):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
    else:
        found = glob.glob("/kaggle/input/**/scripts/run_pipeline.py", recursive=True)
        assert found, "Code not found: set REPO_URL or add the project folder as a Kaggle dataset"
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), REPO)
os.chdir(REPO)
print("code:", REPO, sorted(os.listdir(REPO)))
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--format=commit %h, %cd: %s"],
                     capture_output=True, text=True).stdout)

def run(cmd):
    # Run a step, stream its output, and stop the notebook with a clear message if it fails.
    # (A plain `!python ...` cell never fails, so later cells used to run on missing files.)
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
    if code == -9 or code == 137:
        raise SystemExit("STOPPED: the process was killed by the system, almost always for running out of "
                         "memory. Progress is checkpointed: rerun this cell and it resumes.")
    if code != 0:
        raise SystemExit(f"STOPPED: step failed with exit code {code} (see the error above).")

In [ ]:
!pip install -q "xgboost>=2.0" networkx pyyaml
import torch, xgboost
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| xgboost", xgboost.__version__)

In [ ]:
import yaml
cfg = yaml.safe_load(open("config.yaml"))
OUT = f"/kaggle/working/reports_{DATASET}"
cfg["data"]["raw_csv"] = RAW
cfg["data"]["processed"] = f"/kaggle/working/{DATASET}.parquet"
cfg["split"]["n_windows"] = N_WINDOWS
cfg["split"]["max_train_windows"] = MAX_TRAIN_WINDOWS
cfg["models"]["xgboost"]["device"] = "cuda" if torch.cuda.is_available() else "cpu"
cfg["models"]["search"]["n_trials"] = N_TRIALS
cfg["models"]["search"]["max_rounds"] = MAX_ROUNDS
if DATASET != "HI-Small":  # keep the GNN within the session time on ~30 windows
    cfg["gnn"].update(seeds=[42, 7], max_epochs=150, patience=15)
cfg["outputs"] = {"reports_dir": OUT, "figures_dir": f"{OUT}/figures"}
CONFIG = f"/kaggle/working/config_{DATASET}.yaml"
yaml.safe_dump(cfg, open(CONFIG, "w"))
print(open(CONFIG).read())

## 1. Pipeline: windows, rules, XGBoost, durability checks
Builds and caches features for every window, then tunes and evaluates. Every slow stage is
checkpointed in `/kaggle/working`, so if the run stops, **just rerun this cell**: it resumes from the
last finished stage (within the same session). Each progress line shows peak RAM.

In [ ]:
resume = os.path.exists(f"{OUT}/xgb_search_trials.csv")
run([sys.executable, "-W", "ignore", "scripts/run_pipeline.py", "--config", CONFIG]
    + (["--reuse-search"] if resume else []))

## 2. GNN experiment on the same windows and test accounts

In [ ]:
run([sys.executable, "-W", "ignore", "scripts/run_gnn.py", "--config", CONFIG, "--device", "auto"])

## 3. Results

In [ ]:
import pandas as pd, json
from IPython.display import Image, Markdown, display
display(Markdown("### Pipeline (XGBoost, rules)"))
display(pd.read_csv(f"{OUT}/model_comparison_test.csv").round(4))
display(Markdown("### GNN vs XGBoost (same test accounts)"))
display(pd.read_csv(f"{OUT}/gnn/model_comparison_gnn.csv").round(4))
display(Markdown("### Paired bootstrap: is the GNN better?"))
display(pd.read_csv(f"{OUT}/gnn/paired_differences_gnn.csv").round(4))
display(Image(f"{OUT}/gnn/figures/alert_efficiency_gnn.png"))
display(Markdown(open(f"{OUT}/gnn/summary_gnn.md").read()))
display(Markdown(open(f"{OUT}/summary.md").read()))

In [ ]:
# Download this zip from the Output panel; it holds every report, chart and findings summary
shutil.make_archive(f"/kaggle/working/aml_results_{DATASET}", "zip", OUT)
print("saved", f"/kaggle/working/aml_results_{DATASET}.zip")